# Transverse shear stresses through the thickness

The FSDT gives constant transverse shear strains through the thickness. The
stresses that satisfy equilibrium, vanish at the faces and are continuous at
the ply interfaces are given by `calc_transverse_shear_stress`, for given
shear forces, and by `calc_equilibrium_transverse_shear`, from the strain
gradients of an actual solution. This tutorial also compares the shear
correction methods. The distribution for given shear forces is the
equilibrium approach of Rohwer [1], the recovery from strain gradients the a
posteriori approach of Noor and Peters [5], see
[Transverse shear stresses](../transverse_shear_stresses.rst) and
[Shear correction](../shear_correction.rst). SI units are used throughout.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from composites import laminated_plate, isotropic_plate

laminaprop = (142e9, 8.7e9, 0.28, 5.1e9, 5.1e9, 3.2e9)
plyt = 0.25e-3

## Stresses for given shear forces

For a cross-ply $[0/90]_s$ laminate and a homogeneous plate of the same
thickness, the stresses of the shear force $Q_x$ are compared with the
parabola $\tau_{xz} = 3 Q_x/(2h) (1 - 4z^2/h^2)$. The arguments are
`(z, Qy, Qx)` and the results `(tau_yz, tau_xz)`.

In [2]:
cp = laminated_plate([0, 90, 90, 0], plyt=plyt, laminaprop=laminaprop)
h = cp.h
Qx = 1e3 # N/m
zs = np.linspace(-h/2, h/2, 201)
tau_xz = [cp.calc_transverse_shear_stress(z, 0., Qx)[1] for z in zs]
tau_yz = [cp.calc_transverse_shear_stress(z, Qx, 0.)[0] for z in zs]
parabola = 3*Qx/(2*h)*(1 - 4*zs**2/h**2)

fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(np.array(tau_xz)/1e6, zs*1e3, label='tau_xz, Qx = %g' % Qx)
ax.plot(np.array(tau_yz)/1e6, zs*1e3, label='tau_yz, Qy = %g' % Qx)
ax.plot(parabola/1e6, zs*1e3, 'k--', label='homogeneous')
for zi in np.linspace(-h/2, h/2, 5):
    ax.axhline(zi*1e3, color='gray', lw=0.5)
ax.set_xlabel('stress [MPa]')
ax.set_ylabel('z [mm]')
ax.legend()
fig.tight_layout()

The stresses vanish at the faces and change slope at the interfaces. The
shear correction of `'rohwer'` is the ratio between the strain energy of
these stresses and the one of constant strains:

In [3]:
print('scf_k13 = A55/Abar55 = %.4f' % cp.scf_k13)
print('scf_k23 = A44/Abar44 = %.4f' % cp.scf_k23)

scf_k13 = A55/Abar55 = 0.7454
scf_k23 = A44/Abar44 = 0.6685


## Angle-ply coupling

In a $[45/-45]_s$ laminate, a shear force $Q_x$ also creates $\tau_{yz}$,
through the bending-twisting coupling $D_{16}$, $D_{26}$. The coupling term
`A45` of `Ats` is not zero, although `Abar45` is.

In [4]:
ap = laminated_plate([45, -45, -45, 45], plyt=plyt, laminaprop=laminaprop)
tau = np.array([ap.calc_transverse_shear_stress(z, 0., Qx) for z in zs])
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.plot(tau[:, 1]/1e6, zs*1e3, label='tau_xz')
ax.plot(tau[:, 0]/1e6, zs*1e3, label='tau_yz')
ax.set_xlabel('stress [MPa]')
ax.set_ylabel('z [mm]')
ax.set_title('Qx = %g N/m' % Qx)
ax.legend()
fig.tight_layout()
print('Ats =\n', ap.Ats)
print('Abar_ts =\n', ap.Abar_ts)

Ats =
 [[3136030.33223062  122955.17981669]
 [ 122955.17981669 3136030.33223062]]
Abar_ts =
 [[4150000.       0.]
 [      0. 4150000.]]


## Stresses recovered from strain gradients

`calc_equilibrium_transverse_shear` integrates the equilibrium equations
from the bottom face with the derivatives of the generalized strains
$\{\varepsilon^{(0)}, \varepsilon^{(1)}\}$, e.g. of an FSDT solution. In
cylindrical bending about $y$, with $M_{xx,x} = Q_x$ and the other
derivatives of the resultants equal to zero, the strain gradients are
`inv(ABD) @ {0, 0, 0, Qx, 0, 0}`, and the recovered stresses coincide with
those of `calc_transverse_shear_stress`.

In [5]:
eps_x = np.linalg.solve(cp.ABD, [0, 0, 0, Qx, 0, 0])
eps_y = np.zeros(6)
z, tyz, txz = cp.calc_equilibrium_transverse_shear(eps_x, eps_y, z=zs)
print('max difference [Pa]:',
      np.abs(txz[0] - np.array(tau_xz)).max())

max difference [Pa]: 4.656612873077393e-10


`calc_aposteriori_energy` gives the transverse shear strain energies of the
recovered stresses, from which the a posteriori factor of Noor and Peters [5],
$k_{13} = Q_x^2 / (2 \bar{A}_{55} U_{13})$, follows:

In [6]:
U13, U23, Qy_r, Qx_r = cp.calc_aposteriori_energy(eps_x, eps_y)
print('resultant of the recovered stresses: Qx = %.6f N/m' % Qx_r[0])
print('a posteriori k13 = %.4f' % (Qx**2/(2*cp.Abar55*U13[0])))
print("'rohwer' scf_k13 = %.4f" % cp.scf_k13)

resultant of the recovered stresses: Qx = 1000.000000 N/m
a posteriori k13 = 0.7454
'rohwer' scf_k13 = 0.7454


## Comparison of the shear correction methods

The methods are selected with the argument `shear_correction`:
`'rohwer'` [1], `'vlachoutsis'` [2], `'birman_bert'` [3], `'thickness_shear'`,
the thickness-shear frequency of Yang, Norris and Stavsky [4], and the
constant 5/6. For a
homogeneous plate most of them give 5/6; they differ for laminates, in
particular for a sandwich with a soft core. `'thickness_shear'` requires the
densities of the plies.

In [7]:
core = (0.1e9, 0.1e9, 0.3, 0.04e9, 0.04e9, 0.04e9)
laminates = {
    'isotropic': dict(stack=[0], plyt=1e-3, laminaprop=(70e9, 0.3),
                      rho=2700.),
    '[0/90]s': dict(stack=[0, 90, 90, 0], plyt=plyt, laminaprop=laminaprop,
                    rho=1600.),
    'sandwich': dict(stack=[0, 90, 0, 90, 0], plyts=[plyt, plyt, 10e-3,
                     plyt, plyt],
                     laminaprops=[laminaprop, laminaprop, core, laminaprop,
                                  laminaprop],
                     rhos=[1600., 1600., 50., 1600., 1600.]),
}
methods = ['rohwer', 'vlachoutsis', 'birman_bert', 'thickness_shear',
           'constant']
print('%-12s' % 'k13' + ''.join('%17s' % m for m in methods))
for name, kwargs in laminates.items():
    k13 = [laminated_plate(shear_correction=m, **kwargs).scf_k13
           for m in methods]
    print('%-12s' % name + ''.join('%17.4f' % k for k in k13))

k13                    rohwer      vlachoutsis      birman_bert  thickness_shear         constant
isotropic              0.8333           0.8333           1.0000           0.8225           0.8333
[0/90]s                0.7454           0.7453           0.8882           0.6788           0.8333
sandwich               0.1010           0.1010           0.1036           0.0959           0.8333


## Thickness-shear factors against finite elements through the thickness

The factors of `'thickness_shear'` are
$\kappa = \omega^2 \bar{I}_{2c} / \bar{A}$, with $\omega$ the lowest non-zero
thickness-shear frequency of the laminate, $\bar{I}_{2c}$ the second moment
of the mass about its centroid and $\bar{A}$ the constant-strain stiffness
$\bar{A}_{55}$ (for `scf_k13`) or $\bar{A}_{44}$ (for `scf_k23`). The library
finds $\omega$ with the exact transfer matrix of each ply. As an independent
check, $\omega$ is computed below with 4000 linear finite elements through
the thickness, for the anti-plane motion $u(z)$ with the shear modulus
$q_{55}$ or $q_{44}$ of each ply in the laminate axes; the finite element
frequency converges as $O(1/n^2)$ with the number $n$ of elements. Three
laminates: a sandwich with a core 1000 times softer than the faces and the
reference surface offset by 0.7 mm, the cross-ply $[0/90/0/90]_s$ and the
unsymmetric $[0/90/45]$ with different ply thicknesses and densities, the
last two of the CFRP of Rohwer [1], Table 1.

In [8]:
import scipy.linalg

def thickness_shear_fe(lam, alpha, nelem=4000):
    # lowest non-zero thickness-shear frequency with linear elements;
    # alpha = 0 uses q55L (direction xz), alpha = 1 uses q44L (yz)
    zi = lam.offset - lam.h/2 + np.concatenate(([0.],
            np.cumsum([ply.h for ply in lam.plies])))
    nodes = [zi[0]]
    G = []
    rho = []
    for k, ply in enumerate(lam.plies):
        n = max(4, int(nelem*ply.h/lam.h))
        nodes += list(np.linspace(zi[k], zi[k+1], n + 1)[1:])
        G += [ply.q55L if alpha == 0 else ply.q44L]*n
        rho += [ply.matlamina.rho]*n
    nodes = np.array(nodes)
    K = np.zeros((nodes.size, nodes.size))
    M = np.zeros((nodes.size, nodes.size))
    for e in range(nodes.size - 1):
        le = nodes[e+1] - nodes[e]
        K[e:e+2, e:e+2] += G[e]/le*np.array([[1, -1], [-1, 1]])
        M[e:e+2, e:e+2] += rho[e]*le/6*np.array([[2, 1], [1, 2]])
    return np.sqrt(np.sort(scipy.linalg.eigh(K, M, eigvals_only=True))[1])

cfrp = (138e9, 9.3e9, 0.3, 4.6e9, 4.6e9, 2.3e9)
cases = {
    'sandwich': dict(stack=[0, 0, 0], plyts=[0.5e-3, 8e-3, 0.5e-3],
                     laminaprops=[(70e9, 0.3), (70e6, 0.3), (70e9, 0.3)],
                     rhos=[2700., 100., 2700.], offset=0.7e-3),
    '[0/90/0/90]s': dict(stack=[0, 90, 0, 90, 90, 0, 90, 0], plyt=0.125e-3,
                         laminaprop=cfrp, rho=1600.),
    '[0/90/45]': dict(stack=[0, 90, 45], plyts=[0.2e-3, 0.5e-3, 0.3e-3],
                      laminaprop=cfrp, rhos=[1600., 1000., 2000.]),
}
print('%-14s %-8s %14s %14s %12s' % ('laminate', 'factor', 'composites',
                                      'FE, n = 4000', 'rel. diff.'))
for name, kwargs in cases.items():
    lam = laminated_plate(shear_correction='thickness_shear', **kwargs)
    I2c = lam.intrhoz2 - lam.intrhoz**2/lam.intrho
    for alpha, label, kappa, Abar in [(0, 'scf_k13', lam.scf_k13, lam.Abar55),
                                      (1, 'scf_k23', lam.scf_k23, lam.Abar44)]:
        omega = thickness_shear_fe(lam, alpha)
        kappa_fe = omega**2*I2c/Abar
        print('%-14s %-8s %14.10f %14.10f %12.2e' % (name, label, kappa,
              kappa_fe, kappa/kappa_fe - 1))

laminate       factor       composites   FE, n = 4000   rel. diff.


sandwich       scf_k13    0.0088620390   0.0088620370     2.25e-07


sandwich       scf_k23    0.0088620390   0.0088620370     2.25e-07


[0/90/0/90]s   scf_k13    0.6703025512   0.6703025811    -4.46e-08


[0/90/0/90]s   scf_k23    0.7986442293   0.7986442783    -6.14e-08


[0/90/45]      scf_k13    0.6822305514   0.6822305767    -3.71e-08


[0/90/45]      scf_k23    0.9101235988   0.9101236682    -7.62e-08


The factors of `'thickness_shear'` and those of the finite elements agree
within 1e-6 for every laminate and direction.

## References

1. Rohwer, K., Improved Transverse Shear Stiffnesses for Layered Finite Elements, DFVLR-FB 88-32, 1988.
2. Vlachoutsis, S., Shear correction factors for plates and shells, International Journal for Numerical Methods in Engineering, 33(7), 1537-1552, 1992. https://doi.org/10.1002/nme.1620330712
3. Birman, V. and Bert, C. W., On the Choice of Shear Correction Factor in Sandwich Structures, Journal of Sandwich Structures and Materials, 4(1), 83-95, 2002. https://doi.org/10.1177/1099636202004001180
4. Yang, P. C., Norris, C. H. and Stavsky, Y., Elastic wave propagation in heterogeneous plates, International Journal of Solids and Structures, 2(4), 665-684, 1966. https://doi.org/10.1016/0020-7683(66)90045-x
5. Noor, A. K. and Peters, J. M., A Posteriori Estimates for the Shear Correction Factors in Multi-layered Composite Cylinders, Journal of Engineering Mechanics, 115(6), 1225-1244, 1989. https://doi.org/10.1061/(asce)0733-9399(1989)115:6(1225)